# 183. Customers Who Never Order

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** database, join, subquery, NULL
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/customers-who-never-order/)

```
Table: Customers            Table: Orders
+-------------+---------+   +-------------+------+
| Column Name | Type    |   | Column Name | Type |
+-------------+---------+   +-------------+------+
| id          | int     |   | id          | int  |
| name        | varchar |   | customerId  | int  |
+-------------+---------+   +-------------+------+
id is the primary key.      id is the primary key.
                            customerId is a reference to id from Customers.
```

Write a solution to find all customers who **never** order anything.

Return the result table in **any order**. The result column must be called `Customers`.

---

### Example

```
Customers:                    Orders:              Output:
+----+-------+                +----+------------+  +-----------+
| id | name  |                | id | customerId |  | Customers |
+----+-------+                +----+------------+  +-----------+
| 1  | Joe   |                | 1  | 3          |  | Henry     |
| 2  | Henry |                | 2  | 1          |  | Max       |
| 3  | Sam   |                +----+------------+  +-----------+
| 4  | Max   |
+----+-------+
```

---

Three different queries answer this, they all look correct, and **one of them silently
returns nothing at all** on data you will absolutely meet in the wild. This is the most
important easy SQL problem there is, and the reason is `NULL`.

## Before you write anything

**1.** Write down the three standard ways to express "rows in A with no match in B":

```sql
WHERE id NOT IN     (SELECT customerId FROM Orders)
LEFT JOIN Orders o ON ...  WHERE o.id IS NULL
WHERE NOT EXISTS    (SELECT 1 FROM Orders o WHERE o.customerId = c.id)
```

All three pass the LeetCode example. Write all three and confirm it.

**2.** **Now the trap, and read this slowly.** Suppose `Orders` contains one row whose
`customerId` is `NULL` - a real thing, from an incomplete import or a cancelled
checkout. Predict what each of the three returns. Then run them.

The `NOT IN` version returns **zero rows**. Not an error, not a warning - the correct
answer minus every single row.

**3.** Work out **why**, from the rule you met in #181. `id NOT IN (3, 1, NULL)` expands
to `id <> 3 AND id <> 1 AND id <> NULL`. That last comparison is never `true` and never
`false` - it is `NULL`, "unknown". And `true AND unknown` is `unknown`, which `WHERE`
does not keep. So no row can ever satisfy it. Write that chain out yourself; it is the
single most valuable paragraph in this folder.

**4.** Which of the three are safe against a `NULL` in the subquery? Check each one, and
say *why* the safe ones are safe - `NOT EXISTS` and `LEFT JOIN` never compare a value
against `NULL`, they ask whether a **row** was found.

**5.** For the `LEFT JOIN` version: after joining, which column do you test for `IS NULL`
to mean "there was no matching order"?

Test both `o.id` and `o.customerId` and you will find **both work** - which is worth
understanding rather than shrugging at. `o.customerId` is safe here for a precise reason:
it is the **join key**, and a row that matched must have had a non-null join key, since
`NULL = c.id` is never true. So a null `o.customerId` after the join can only mean "no
match". The order row with the null `customerId` never joins to anybody, so it cannot
produce a false positive either.

Now state the rule that generalises: testing the **join key** or a **primary key** for
`IS NULL` is always safe; testing any *other* nullable column of the right-hand table is
not, because a genuinely matched row could have a null in it. Which column would you
write out of habit, so that you are safe even in the query where it matters?

**6.** How would you make `NOT IN` safe if you had to keep it? There is a one-clause fix.
Write it - and then say why "just add a filter" is a worse habit than picking the
construct that cannot fail.

## Two routes

**A - `LEFT JOIN` and keep the misses** *(write this first)*

```sql
SELECT c.name AS Customers
FROM Customers c
LEFT JOIN Orders o ON o.customerId = c.id
WHERE o.id IS NULL
```

This is #175's `LEFT JOIN` used the other way round. There, you kept every left row and
showed the nulls; here, you keep every left row and then keep **only** the nulls - which
are exactly the customers with no matching order. Testing `o.id` (a primary key) or
`o.customerId` (the join key) both work - see question 5 for why, and for the rule that
tells you when the choice would matter.

**B - `NOT EXISTS`**

```sql
SELECT c.name AS Customers
FROM Customers c
WHERE NOT EXISTS (SELECT 1 FROM Orders o WHERE o.customerId = c.id)
```

Reads exactly like the English, is immune to the `NULL` problem by construction, and
lets the engine stop as soon as it finds one matching order. `SELECT 1` is idiomatic -
`EXISTS` only cares whether a row came back, never what was in it.

**And the one to avoid:** `NOT IN (SELECT customerId FROM Orders)`. It is the most
natural sentence in English and the most dangerous one in SQL. There is a test case
below that contains exactly one `NULL` `customerId`, and it exists to make this version
fail in front of you.

> **`NOT IN` with a nullable subquery is a silent, total failure.** It does not error.
> It does not warn. It returns zero rows and your report says "no customers have ever
> failed to order", which someone will believe. Prefer `NOT EXISTS` or a `LEFT JOIN` -
> not because `NOT IN` is always wrong, but because the failure mode of being wrong is
> invisible.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Customers (id INTEGER, name TEXT);
CREATE TABLE Orders (id INTEGER, customerId INTEGER);"""

EXPECTED_COLUMNS = ['Customers']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Customers VALUES (1,'Joe'), (2,'Henry'), (3,'Sam'), (4,'Max');
INSERT INTO Orders VALUES (1,3), (2,1);
''', [('Henry',), ('Max',)])

check("everybody has ordered", '''
INSERT INTO Customers VALUES (1,'A'), (2,'B');
INSERT INTO Orders VALUES (1,1), (2,2);
''', [])

check("nobody has ordered", '''
INSERT INTO Customers VALUES (1,'A'), (2,'B');
''', [('A',), ('B',)])

check("*** question 2: an order with a NULL customerId - NOT IN dies here ***", '''
INSERT INTO Customers VALUES (1,'Joe'), (2,'Henry'), (3,'Sam');
INSERT INTO Orders VALUES (1,3), (2,NULL);
''', [('Joe',), ('Henry',)])

check("*** every order has a NULL customerId ***", '''
INSERT INTO Customers VALUES (1,'A'), (2,'B');
INSERT INTO Orders VALUES (1,NULL), (2,NULL);
''', [('A',), ('B',)])

check("one customer ordering many times", '''
INSERT INTO Customers VALUES (1,'Busy'), (2,'Quiet');
INSERT INTO Orders VALUES (1,1), (2,1), (3,1), (4,1);
''', [('Quiet',)])

check("an order for a customer who does not exist", '''
INSERT INTO Customers VALUES (1,'A');
INSERT INTO Orders VALUES (1,99);
''', [('A',)])

check("two customers with the same name", '''
INSERT INTO Customers VALUES (1,'Sam'), (2,'Sam');
INSERT INTO Orders VALUES (1,1);
''', [('Sam',)])

check("no customers at all", '''
INSERT INTO Orders VALUES (1,1);
''', [])

check("both tables empty", '', [])

## After it passes

- **Make `NOT IN` fail, and watch it.** Write the `NOT IN` version, run it with `show`
  against the NULL-customerId dataset, and see zero rows come back with no error. Then
  run it against the LeetCode dataset and watch it be perfectly correct. A bug that only
  appears when the data is imperfect is the worst kind there is, and now you have seen
  one.
- **Fix it two ways.** Add `WHERE customerId IS NOT NULL` inside the subquery, and check
  it passes. Then delete the whole thing and use `NOT EXISTS`. Write one sentence on why
  the second fix is better even though both work.
- **Test the wrong column.** Change your `LEFT JOIN` version to `WHERE o.customerId IS
  NULL` and run it against the NULL-customerId dataset. Work out exactly which row it
  wrongly returns, and why testing a nullable column for "no match" is a bug waiting for
  the right data.
- **Then look at all three plans.** `EXPLAIN QUERY PLAN` for each of the three. They may
  well be identical - modern planners rewrite anti-joins into one shape. Which means the
  choice between them is about **correctness and readability, not speed** - a good thing
  to have proved to yourself rather than assumed.
- Siblings: **#175 Combine Two Tables** (the `LEFT JOIN` this inverts), #181 Employees
  Earning More Than Their Managers (`NULL` comparison helping instead of hurting),
  #1148 Article Views I, #607 Sales Person (the same anti-join, more tables).